# Step 3.6: the session-aware indicators

This notebook is the check of step 3.6. `ff.indicators` now holds the indicators intraday futures traders read by the session: the session's VWAP, the opening range, the session before, relative volume, cumulative delta, swings, and more. Here you plot one day's session VWAP and its opening range on 5-minute bars: the VWAP starts again at the session's open, 17:00 Chicago the evening before its trade date, and the range appears once its window has closed.

Pick the `.venv` kernel and run all cells. It reads the store in `FEFAT_DATA_DIR` and keeps the values in the feature cache under `FEFAT_HOME`.

## One day of 5-minute bars

The product is MES when your store holds its 1-minute bars, else the product with the most contracts, and the day the last trade date of its data whose bars reach 14:00 Chicago.

Expected: the product, the day, and its contract.

In [ ]:
from datetime import datetime, time, timedelta
from zoneinfo import ZoneInfo

import matplotlib.pyplot as plt
import polars as pl

import fefat as ff
from fefat.market.engine_units import price_text

listed = ff.market.datasets().filter(pl.col("schema") == "ohlcv-1m")
chosen = listed.sort(
    pl.col("root") != "MES", "contracts", descending=[False, True], maintain_order=True
).row(0, named=True)
ROOT, LAST = chosen["root"], chosen["last_trade_date"]
slots = {
    "session_vwap": ff.indicators.session_vwap(),
    "opening": ff.indicators.opening_range(minutes=30, anchor="08:30"),
}
computed = ff.features.compute(slots, ROOT, "5m", LAST - timedelta(days=10), LAST).collect()
chicago = {
    name: pl.col(name).cast(pl.Datetime("ns", "UTC")).dt.convert_time_zone("America/Chicago")
    for name in ("ts_open", "ts_close")
}
bars = computed.with_columns(opens=chicago["ts_open"], closes=chicago["ts_close"])
at_two = bars.filter(pl.col("opens").dt.strftime("%H:%M") == "14:00")
DAY = at_two.get_column("trade_date").max()
day = bars.filter(pl.col("trade_date") == DAY)
print(f"{ROOT} 5-minute bars of {DAY}, contract {day.item(0, 'contract')}")

## The VWAP starts again at the open

The session's first bar opens at 17:00 Chicago the evening before its trade date. A VWAP that starts again there equals the price of the session's first bar that traded: its vwap, or, where your data has none, its typical price, the mean of its high, low, and close, as charting platforms weigh a bar.

Expected: when that bar opens, its VWAP and its own price, the same, and `PASSED`.

In [ ]:
first = day.filter(pl.col("volume") > 0).row(0, named=True)
typical = (first["high"] + first["low"] + first["close"]) / 3
own = typical if first["vwap"] is None else first["vwap"]
gap = abs(first["session_vwap"] - own)
verdict = "PASSED" if gap <= 1e-3 else "FAILED"
opened = first["opens"].strftime("%Y-%m-%d %H:%M")
vwap, price = first["session_vwap"] / 1e9, own / 1e9
print(f"First bar that traded opens {opened} Chicago: VWAP {vwap:.2f}, its own price {price:.2f}")
print(f"The VWAP started again at the session's open: {verdict}")

## The opening range

The opening range of the 30 minutes from 08:30 Chicago, the cash open, is the high and the low of the bars that traded inside it. It appears on the bar closing at 09:00, once the window's last bar has closed, and is null on every bar closing before it, the evening's included.

Expected: the range's high, low, and width; the number of bars closing before 09:00 that have one, 0, and of those closing from 09:00 on that do, all of them, and `PASSED`; then a chart of the day's closes with the VWAP and the range's high and low.

In [ ]:
nine = datetime.combine(DAY, time(9), ZoneInfo("America/Chicago"))
early = day.filter(pl.col("closes") < nine)
late = day.filter(pl.col("closes") >= nine)
held = late.row(0, named=True)
high, low = price_text(held["opening_range_high"]), price_text(held["opening_range_low"])
width = price_text(held["opening_range_width"])
print(f"Opening range of 08:30 to 09:00 Chicago: high {high}, low {low}, width {width}")
set_early = early.get_column("opening_range_high").drop_nulls().len()
set_late = late.get_column("opening_range_high").drop_nulls().len()
verdict = "PASSED" if set_early == 0 and set_late == late.height else "FAILED"
counts = f"{set_early}; from 09:00: {set_late} of {late.height}"
print(f"Bars closing before 09:00 with a range: {counts}: {verdict}")

figure, axes = plt.subplots(figsize=(10, 4))
at = day.get_column("closes").dt.replace_time_zone(None)
axes.plot(at, day.get_column("close") / 1e9, linewidth=1, label="close")
axes.plot(at, day.get_column("session_vwap") / 1e9, label="session VWAP")
for line in ("high", "low"):
    level = day.get_column(f"opening_range_{line}") / 1e9
    axes.plot(at, level, linestyle="--", label=f"opening range {line}")
axes.set_title(f"{ROOT} on {DAY}: 5-minute bars, Chicago time")
axes.legend()
plt.show()